# Interactive plots: the `bloch_schrodinger.plotting` module (tutorial with exercises)

This tutorial goes through the interactive plotting functions of the package on a single example: an optical honeycomb lattice with a tunable depth and sublattice imbalance. The sections are:

1. potential landscapes,
2. cuts through the potential,
3. eigenvalue landscapes (band structures, and quantities derived from them),
4. eigenvectors,
5. cuts through the eigenvectors,
6. eigenvalues and eigenvectors together,
7. styling plots with templates,
8. Wannier functions, with the lattice sites marked.

All the plotting functions work the same way: **every dimension of the data that is not plotted becomes a slider.** You never build the sliders yourself. Whatever parameters you created with `create_parameter`, and whatever k-points you solved on, show up as sliders automatically.

The spatial axes to plot against are chosen with `cart_axes`, where `0 = x`, `1 = y` and `2 = z`. Two axes give a 2D map, one axis gives a cut, and any spatial axis left out becomes a slider as well. That slider sets where the cut sits.

### How to use this notebook

Each section starts with an **explanation and a worked example**. Run these cells in order, because later sections reuse their variables. After the example comes a ✏️ **exercise**:

- The exercise cell holds commented-out starting code. Uncomment it and fill in the `...`.
- The **solution** is in the collapsed cell right after it. In VS Code, click the `…` placeholder (or the bar to the left of the cell) to expand it. In JupyterLab, click the blue bar to the left of the cell.
- Each exercise depends only on the worked examples, not on other exercises, so you can skip any of them.
- **Run All** works. The unfinished exercise cells do nothing, and the solution cells still run, so you can look at the target figure before you look at the code.

## Overview: the plotting functions

Every function takes the xarray DataArrays the solvers return. **Every dimension that is not plotted becomes a slider**, so one call covers a whole parameter scan. Run the notebook with `%matplotlib widget`.

**High level: one call, one interactive figure**

| Function | Draws | Required inputs | Use it for |
|---|---|---|---|
| `Potential.plot(cart_axes=[0, 1], show_minima=False)` | the potential as a map, or a line cut with `cart_axes=[0]` | a `Potential` | a first look at a lattice; `show_minima=True` marks the sites |
| `plot_cuts(da, dim, groupby=['band'])` | lines of `da` against one dimension, one line per value of the `groupby` dims | any DataArray that has `dim` as a coordinate | band structures (`eigva` against `ky` or a path), and any derived quantity against a parameter: gaps, widths, spreads, U, V |
| `plot_eigenvector(plots, potentials, templates=None, cart_axes=[0, 1], show_minima=False)` | a grid of maps (or line cuts) of fields, with the potential's contours on top | `plots`: a matrix (list of lists) of DataArrays on the potential's lattice; `potentials`: a matrix of the same shape of `Potential` or `None` | eigenvectors, Wannier functions, any field on the lattice; several panels side by side on shared sliders |
| `dashboard(eigva, eigvadim, eigveplots, potential, template)` | the bands next to a grid of eigenvector maps, linked | the eigenvalues, the dim to plot them against, a matrix of eigenvector DataArrays, one `Potential`, a template | which state belongs to which band, as you move along k or a parameter |
| `energy_levels(eigva, eigve, potential)` | each mode's profile drawn at its energy, over a cut through the potential (direction set by the `offset` and `rotation` sliders) | the eigenvalues and eigenvectors of one solve, and its `Potential` | bound states in a trap or a well: which level looks like what |
| `plot_isosurface(volume, color=None, potential=None)` | a 3D level surface (plotly) | a real field with three spatial dims (`a1`, `a2`, `a3`) | 3D modes: `abs(eigve)` for the shape, `np.angle(eigve)` for the colour (`cyclic=True`) |

**Templates: the style of a map**

| Function | Returns | Use |
|---|---|---|
| `cmesh_tmpl(name)` | a colour-map template: `'amplitude'`, `'amplitude - log'`, `'real'`, `'real - log'` or `'phase'` | `templates=` of `plot_eigenvector`, `template=` of `dashboard` and `create_map`. For a fixed colour range, set `t['autoscale'] = False` and `t['clim'] = (lo, hi)` |
| `contour_tmpl(lvls=3)`, `quiver_tmpl()` | contour and arrow templates | combined as `(cmesh_tmpl('real'), contour_tmpl(6), quiver_tmpl())` |
| `get_template(name)` | the complete template `plot_eigenvector` uses for a name | a starting point to modify |

**Lattice geometry: sites where they really are** (`bloch_schrodinger.lattice`, [section 9](#9.-The-lattice-geometry:-snapped-sites-and-a-schematic-overlay))

| Function | Does | Required inputs | Use it for |
|---|---|---|---|
| `LatticeGeometry.hexagonal(a1, a2, origin)` (also `square`, `checkerboard`, `kagome`, `lieb`, or your own `LatticeGeometry(a1, a2, sites)`) | the lattice vectors and the named sites A, B, ... at their exact positions | the potential's lattice vectors, and where site A is | everything below; set it as `potential.geometry` for `lattice=True` |
| `potential.find_sites(geometry)` | the minima of `find_minima`, moved exactly onto the sites within a tolerance, and labelled | a 2D `Potential` and a geometry | which sites are wells, at exact positions |
| `lattice=True` in `Potential.plot`, `plot_eigenvector`, `dashboard` | the lattice drawn schematically over the maps: labelled circles at the occupied sites, optionally bonds, empty sites, unit cell; follows the sliders | a potential with a `geometry`, or `lattice=geometry` | publication-style maps of fields and Wannier functions |
| `draw_lattice(ax, geometry, occupied)` | the same overlay on any `Axes` | an `Axes` with its limits set | figures of your own |

**Low level: your own figure**

| Function | Draws into an existing `Axes` | Required inputs | Use it for |
|---|---|---|---|
| `create_map(fig, ax, cart_axes, data, method)` | `'pcolormesh'`, `'contour'` or `'contourf'` | a DataArray with dims `a1`, `a2` and `x`, `y` coordinates. Any 2D grid works, e.g. a $(\lambda, \theta)$ plane renamed to `a1`, `a2` | layouts the functions above do not cover |
| `create_line(fig, ax, cart_axis, data)` | a line | a DataArray on the lattice | a spatial cut in a figure of your own |
| `create_quiver(fig, ax, cart_axes, dataU, dataV)` | arrows | the two components of a vector field | currents, polarization or spin textures |

Each low-level function returns `(sliders, update, ax)`. Show the sliders with `display(VBox(list(sliders.values()) + [interactive_output(update, sliders)]))`. Several plots can share one set of sliders through an update function that calls each plot's `update`.

## The honeycomb lattice

We use the same lattice as in the [Wannier](Wannier.ipynb) tutorial: three laser beams at 120° create a honeycomb lattice. The lattice depth `s` and the sublattice imbalance `delta` are parameters, so each of them gets a slider in every plot below.

In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from bloch_schrodinger.potential import Potential, create_parameter

# Creating the optical lattice, see https://journals.aps.org/pra/abstract/10.1103/PhysRevA.80.043411 for more information on the lattice.
## The lattice parameters
kl = 2
m = 1
hbar = 1
E_r = hbar**2 * kl**2 / 2 / m
delta = create_parameter('delta', np.linspace(-0.2, 0.2, 5)) # sublattice imbalance
s = create_parameter('s', np.linspace(3, 9, 3)) # lattice depth, in recoil energies

V1 = -(s - delta) * E_r
V2 = -(s + delta) * E_r

## The lattice geometry
a = 4*np.pi / 3**1.5 / kl
a1 = np.array([3 * a/2, -(3**0.5) * a/2])  # 1st lattice vector
a2 = np.array([3 * a/2, 3**0.5 * a/2])  # 2nd lattice vector

k1 = kl * np.array([-3**0.5 / 2 , 1/2])
k2 = kl * np.array([3**0.5 / 2 , 1/2])
k3 = kl * np.array([0,-1])

K = np.array([0, 4*np.pi/3**1.5/a]) # The K point of the Brillouin zone

## Creating the potential object
honeycomb = Potential(
    unitvecs = [a1, a2],
    resolution = (48, 48),
    v0 = 0,
)

dirs = [k[0] * (honeycomb.x - a1[0]) + k[1] * honeycomb.y for k in (k1, k2, k3)]

for i in range(3):
    honeycomb.add(2*V1*np.cos((dirs[i-1]-dirs[i]) - 2*np.pi/3)/2)
    honeycomb.add(2*V2*np.cos((dirs[i-1]-dirs[i]) + 2*np.pi/3)/2)

print(honeycomb.V.dims)

### ✏️ Exercise 0 (getting to know the data)

Before plotting anything, look at the data that the sliders will be built from.

1. Print the values of the `s` and `delta` coordinates of `honeycomb.V`, and its `sizes`. How many positions will the `s` and `delta` sliders have?
2. The two sites of the unit cell sit at the lattice coordinates $(a_1, a_2) = (-1/6, -1/6)$ and $(1/6, 1/6)$. Use `.sel` and `.interp` to get the potential at both sites for `s = 9`, `delta = 0.2`. Which site is deeper? Does this change for `delta = -0.2`?

<details><summary>Hint</summary>

`honeycomb.V.sel(s=9, delta=0.2)` selects one parameter point, and `.interp(a1=..., a2=...)` evaluates it at a point inside the cell. Wrap the result in `float(...)` to print a plain number.
</details>

In [ ]:
# Exercise 0: your code here
# print(honeycomb.V.coords['s'].values)
# print(...)
# print(honeycomb.V.sizes)
#
# for d in [0.2, -0.2]:
#     V_pt = honeycomb.V.sel(s=9, delta=d)
#     V_A = float(V_pt.interp(a1=..., a2=...))
#     V_B = ...
#     print(d, V_A, V_B)

In [ ]:
# Solution to exercise 0
print('s     :', honeycomb.V.coords['s'].values)       # 3 positions
print('delta :', honeycomb.V.coords['delta'].values)   # 5 positions
print(dict(honeycomb.V.sizes))

for d in [0.2, -0.2]:
    V_pt = honeycomb.V.sel(s=9, delta=d)
    V_A = float(V_pt.interp(a1=-1/6, a2=-1/6))
    V_B = float(V_pt.interp(a1=1/6, a2=1/6))
    deeper = 'A = (-1/6, -1/6)' if V_A < V_B else 'B = (1/6, 1/6)'
    print(f'delta = {d:+.1f}:  V_A = {V_A:.2f},  V_B = {V_B:.2f}  ->  deeper site: {deeper}')

## 1. Potential landscapes

The quickest option is the `plot` method of the `Potential` class. `show_minima=True` marks the local minima of the potential, here the two lattice sites A and B of the unit cell. Move the `delta` slider to see one site get deeper than the other.

In [ ]:
honeycomb.plot(show_minima=True)
plt.show()

A single unit cell is not very telling. `tile` returns a copy of the potential repeated over several unit cells, which is much easier to read. The two pairs give the range of cells along `a1` and `a2`.

In [ ]:
tiled = honeycomb.tile([[-1, 2], [-1, 2]])
tiled.plot()
plt.show()

### ✏️ Exercise 1.1

Tile the potential over **5 × 5** unit cells, from cell $-2$ to cell $3$ along both lattice vectors. Plot it with the lattice sites marked and with the `'viridis'` colormap.

*Extra keyword arguments of `Potential.plot` are passed on to `pcolormesh`.*

Move the `delta` slider from $-0.2$ to $0.2$. Which of the marked sites become deeper?

In [ ]:
# Exercise 1.1: your code here
# big = honeycomb.tile(...)
# big.plot(show_minima=..., cmap=...)
# plt.show()

In [ ]:
# Solution to exercise 1.1
big = honeycomb.tile([[-2, 3], [-2, 3]])
big.plot(show_minima=True, cmap='viridis')
plt.show()
# At delta = 0 the two sublattices are equivalent. For delta != 0, one of them (one site of every unit
# cell) becomes deeper, and the sign of delta selects which one.

For more control over the style, pass the potential's DataArray `.V` to `plot_eigenvector`. This is the package's general plotting function for fields, which we will use for the eigenvectors below. It takes **matrices** (lists of lists) of fields, of potentials drawn as contour overlays, and of templates. Each entry of these matrices is one subplot of the figure.

The templates set the style of each subplot. For now, it is enough to know that a name such as `'real'` picks a preset style, and a tuple `(colormesh, contour, quiver)` styles the three layers of a subplot separately. [Section 7](#7.-Styling-plots-with-templates) covers templates in detail.

Here the left panel shows the potential with 6 dashed contour levels. The right panel has no colormesh (`None` in `plots`): it shows only the contours, with the optical dipole force $\mathbf{F} = -\nabla V$ overlaid as arrows through the `quivers` argument. The force is computed from the derivatives along the lattice axes $a_1$ and $a_2$, converted to cartesian components.

In [ ]:
from bloch_schrodinger.plotting import plot_eigenvector, contour_tmpl

# dV/da_i = a_i . grad(V), so grad(V) = A^-1 (dV/da_1, dV/da_2), with A the matrix of the lattice vectors
A_inv = np.linalg.inv(np.array([a1, a2]))
dV1 = tiled.V.differentiate('a1')
dV2 = tiled.V.differentiate('a2')
Fx = -(A_inv[0, 0] * dV1 + A_inv[0, 1] * dV2)
Fy = -(A_inv[1, 0] * dV1 + A_inv[1, 1] * dV2)

force_template = {
    # passed to plt.quiver: with scale_units='xy', an arrow of length 1 represents a force of 'scale'
    'fkwargs': {'color': 'C3', 'pivot': 'mid', 'angles': 'xy', 'scale_units': 'xy', 'scale': 120, 'width': 0.006},
    'density': 12, # only one arrow every 12 grid points
}

fig, axs = plot_eigenvector(
    plots = [[tiled.V, None]], # None: no colormesh in the second panel, only contours and arrows
    potentials = [[tiled, tiled]],
    templates = [[('real', contour_tmpl(6)), (None, contour_tmpl(6), force_template)]],
    quivers = [[None, (Fx, Fy)]],
)
axs[0][0].set_title('$V(x, y)$')
axs[0][1].set_title(r'$\mathbf{F} = -\nabla V$')
plt.show()

### ✏️ Exercise 1.2

Make a **single-panel** figure that shows all three layers at once:

- the colormesh of `tiled.V` with the `'real'` preset,
- **4** dashed contour levels of the potential,
- the force field `(Fx, Fy)` from the example above, drawn as **orange** arrows (`'C1'`) with **one arrow every 8 grid points**.

Title the panel `'$V$ and $-\nabla V$'`. Check that the arrows point towards the minima of the potential.

<details><summary>Hint</summary>

A subplot with all three layers takes a 3-tuple `(colormesh, contour, quiver)` in `templates`, and its arrows go in `quivers=[[(Fx, Fy)]]`. Start from a copy of `force_template` and change `color` and `density`.
</details>

In [ ]:
# Exercise 1.2: your code here
# arrows = {
#     'fkwargs': {...},
#     'density': ...,
# }
# fig, axs = plot_eigenvector(
#     plots = [[...]],
#     potentials = [[...]],
#     templates = [[(..., ..., ...)]],
#     quivers = [[...]],
# )
# axs[0][0].set_title(...)
# plt.show()

In [ ]:
# Solution to exercise 1.2
arrows = {
    'fkwargs': {'color': 'C1', 'pivot': 'mid', 'angles': 'xy', 'scale_units': 'xy', 'scale': 120, 'width': 0.006},
    'density': 8,
}

fig, axs = plot_eigenvector(
    plots = [[tiled.V]],
    potentials = [[tiled]],
    templates = [[('real', contour_tmpl(4), arrows)]],
    quivers = [[(Fx, Fy)]],
)
axs[0][0].set_title(r'$V$ and $-\nabla V$')
plt.show()

## 2. Cuts through the potential

Giving a single axis to `cart_axes` plots a cut instead of a map. The other spatial axis becomes a slider: with `cart_axes=[0]` the cut runs along `x`, and the `y` slider moves it across the lattice.

In [ ]:
tiled.plot(cart_axes=[0])
plt.show()

`plot_eigenvector` accepts the same argument, either as a single list for every subplot, or as a matrix of the same shape as `plots` to give each subplot its own axes. Here we draw a cut along `x` next to a cut along `y`.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[tiled.V, tiled.V]],
    potentials = [[None, None]],
    cart_axes = [[[0], [1]]],
)
axs[0][0].set_title('cut along x')
axs[0][1].set_title('cut along y')
plt.show()

### ✏️ Exercise 2.1

Make a **1 × 3** figure of `tiled.V`:

1. the full map, with the `'real'` preset and contours,
2. a cut along `x`,
3. a cut along `y`.

Give each panel a title. Then use the `y` slider to find a cut along `x` that passes through both sites of the cell. (In the cartesian frame of this lattice the two sites sit at $(\pm a/2, 0)$.)

<details><summary>Hint</summary>

`cart_axes` takes a matrix shaped like `plots`: `[[[0, 1], [0], [1]]]`. The cuts need no template (`None`), and use `None` in `potentials` for them.
</details>

In [ ]:
# Exercise 2.1: your code here
# fig, axs = plot_eigenvector(
#     plots = [[tiled.V, tiled.V, tiled.V]],
#     potentials = [[..., ..., ...]],
#     templates = [[..., ..., ...]],
#     cart_axes = [[..., ..., ...]],
# )
# for ax, title in zip(axs[0], [...]):
#     ax.set_title(title)
# plt.show()

In [ ]:
# Solution to exercise 2.1
fig, axs = plot_eigenvector(
    plots = [[tiled.V, tiled.V, tiled.V]],
    potentials = [[tiled, None, None]],
    templates = [['real', None, None]],
    cart_axes = [[[0, 1], [0], [1]]],
)
for ax, title in zip(axs[0], ['map', 'cut along x', 'cut along y']):
    ax.set_title(title)
plt.show()
# With y close to 0 the cut along x passes through both sites at x = -a/2 and x = +a/2.

## Solving the band structure

We now compute the bands along a line through the Brillouin zone, from $-1.5K$ to $1.5K$ along $k_y$. This line passes through $\Gamma$ and both Dirac points $\pm K$. With a single `kx` value the band structure is a function of `ky`, `s` and `delta`.

We use the **plane-wave solver** `PWSolver`. A smooth potential such as an optical lattice needs only a few plane waves, so this is fast and accurate. The cut-off `E_lim` sets which plane waves are kept: those with kinetic energy $\alpha|G|^2$ below it. A few times the lattice depth is enough; here `E_lim = 300` keeps 187 plane waves and the energies are converged to $10^{-9}$. The finite-difference `FDSolver` on the 48 × 48 grid is about 4 times slower and off by $\sim 0.03$. That solver is the one to use for potentials with sharp edges or for several coupled fields.

The plane-wave solver returns its eigenvectors as plane-wave coefficients. `compute_u` turns them into the mode profiles on the lattice grid, which is what the plotting functions take. Their global phase is arbitrary, so we fix it as the `FDSolver` does, real and positive at one point of the cell, so that real parts and phases do not jump from one slider position to the next.

That is 555 diagonalizations (37 k-points × 3 depths × 5 imbalances), so we parallelize the solve.

In [ ]:
from bloch_schrodinger.pwsolver import PWSolver

solver = PWSolver(honeycomb, alpha = hbar**2/2/m, E_lim = 300)
solver.create_reciprocal_grid([0, np.linspace(-1.5*K[1], 1.5*K[1], 37)])

eigva, eigve_pw = solver.solve(n_eigva = 4, parallel = True)

# the mode profiles on the lattice grid
eigve = solver.compute_u(eigve_pw)
# fix the arbitrary global phase: real and positive near the corner of the cell
eigve = eigve * xr.ufuncs.exp(-1j * xr.ufuncs.angle(eigve.sel(a1 = 0.01, a2 = 0.01, method = 'nearest')))
# the cartesian coordinates the plotting functions read
eigve = eigve.assign_coords(x = honeycomb.x, y = honeycomb.y).transpose('band', ..., 'a1', 'a2')

print(eigva.dims) # kx only takes a single value, so it is not a dimension (and won't get a slider)
print(eigve.dims)

## 3. Eigenvalue landscapes

`plot_cuts` plots a DataArray against one of its dimensions, and draws one line per value of the dimensions listed in `groupby` (`['band']` by default). Here that gives the band structure along `ky`, with `s` and `delta` as sliders. At `delta = 0` the two lowest bands touch at the Dirac points $\pm K$ (dotted lines). Any imbalance opens a gap there.

In [ ]:
from bloch_schrodinger.plotting import plot_cuts

fig, ax = plot_cuts(eigva, 'ky')
for k in [-K[1], K[1]]:
    ax.axvline(k, color='gray', linestyle='dotted')
plt.show()

The styles in `linekws` are cycled over the bands, and `ymin`/`ymax` fix the vertical range, which otherwise follows the sliders. Selecting before plotting reduces the number of lines and sliders: here we keep only the two lowest bands.

In [ ]:
fig, ax = plot_cuts(
    eigva.sel(band=[0, 1]), 'ky',
    linekws = [{'color': 'C0'}, {'color': 'C3', 'linestyle': '--'}],
    ymin = -20, ymax = 5,
)
plt.show()

`plot_cuts` works on any DataArray, not only on the raw eigenvalues. Quantities derived with ordinary xarray operations are plotted the same way. Here the gap at the Dirac point is plotted against `delta` with `groupby=[]`, since it is a single curve, and the width of the lowest band against `s`.

In [ ]:
gap_K = (eigva.sel(band=1) - eigva.sel(band=0)).sel(ky=K[1], method='nearest').rename('gap at K')
fig, ax = plot_cuts(gap_K, 'delta', groupby=[])
plt.show()

bandwidth = (eigva.max('ky') - eigva.min('ky')).rename('bandwidth')
fig, ax = plot_cuts(bandwidth, 's')
plt.show()

### ✏️ Exercise 3.1

**Zoom in on the Dirac point** $K$. Plot the two lowest bands, only for $k_y$ within $\pm 0.6$ of `K[1]`. Use a different color for each band, and add markers (`'marker': 'o'`) so that you can see the k-points of the grid. Mark $K$ with a dotted vertical line.

Move `delta` away from 0: how does the gap at $K$ open?

<details><summary>Hint</summary>

Select the range before plotting, with `ky=slice(K[1] - 0.6, K[1] + 0.6)` inside `.sel`. Then the automatic y-range follows the zoomed data only. Passing `xmin`/`xmax` instead would zoom in on the x-axis, but the y-range would still fit the whole band.
</details>

In [ ]:
# Exercise 3.1: your code here
# zoom = eigva.sel(band=..., ky=slice(..., ...))
# fig, ax = plot_cuts(zoom, 'ky', linekws=[{...}, {...}])
# ax.axvline(...)
# plt.show()

In [ ]:
# Solution to exercise 3.1
zoom = eigva.sel(band=[0, 1], ky=slice(K[1] - 0.6, K[1] + 0.6))
fig, ax = plot_cuts(
    zoom, 'ky',
    linekws = [{'color': 'C0', 'marker': 'o'}, {'color': 'C3', 'marker': 'o'}],
)
ax.axvline(K[1], color='gray', linestyle='dotted')
plt.show()
# At delta = 0 the two bands touch at K with a linear (Dirac) crossing. Any imbalance opens a gap
# exactly at K, and the bands become rounded there.

### ✏️ Exercise 3.2

For every `(s, delta)`, compute the **smallest direct gap** between bands 0 and 1 over the whole `ky` line. Plot it against `delta`, with **one line per lattice depth `s`** and a legend.

- Is the gap symmetric in `delta`? Where does it close?
- Does a deeper lattice make the gap larger or smaller at a given `delta`?

<details><summary>Hint</summary>

`.min('ky')` reduces over the k-points. `groupby=['s']` draws one line per value of `s`, and since no dimension is left over, the plot has no slider. Put the legend labels in `linekws`, e.g. `[{'label': f's = {v:g}'} for v in min_gap.s.values]`.
</details>

In [ ]:
# Exercise 3.2: your code here
# min_gap = (... - ...).min(...).rename('smallest gap')
# fig, ax = plot_cuts(min_gap, 'delta', groupby=[...], linekws=[...])
# ax.legend()
# plt.show()

In [ ]:
# Solution to exercise 3.2
min_gap = (eigva.sel(band=1) - eigva.sel(band=0)).min('ky').rename('smallest gap')
fig, ax = plot_cuts(
    min_gap, 'delta', groupby=['s'],
    linekws = [{'label': f's = {v:g}', 'marker': 'o'} for v in min_gap.s.values],
)
ax.legend()
plt.show()
print(min_gap.round(3).to_pandas())
# The gap is symmetric in delta and closes at delta = 0 (the small leftover is the finite k-grid). At a given delta, a deeper lattice gives a larger gap.

## 4. Eigenvectors

The eigenvectors are plotted with `plot_eigenvector`, exactly like the potential above. Plot real quantities: the density `abs(eigve)**2`, the real part `eigve.real`, or the phase. For the phase, use `xr.ufuncs.angle(eigve)` rather than `np.angle`, which returns a bare numpy array that lost its dimensions and coordinates.

Every other dimension becomes a slider: the band, the k-point and the lattice parameters. Move `ky` to the K point (`ky ≈ 2.0`) to see the phase wind around the lattice sites.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(eigve)**2, eigve.real, xr.ufuncs.angle(eigve)]],
    potentials = [[honeycomb, honeycomb, honeycomb]],
    templates = [['amplitude', 'real', 'phase']],
)
axs[0][0].set_title(r'$|\psi|^2$')
axs[0][1].set_title(r'Re$(\psi)$')
axs[0][2].set_title(r'arg$(\psi)$')
plt.show()

### ✏️ Exercise 4.1

Look at the states **at the K point** only. Make a **2 × 2** grid:

| | band 0 | band 1 |
|---|---|---|
| top row | density $|\psi|^2$ | density $|\psi|^2$ |
| bottom row | phase $\arg\psi$ | phase $\arg\psi$ |

Since `band` and `ky` are both fixed, only `s` and `delta` remain as sliders. Set `delta = 0.2`, then `delta = -0.2`: on which sublattice does each band sit?

<details><summary>Hint</summary>

`psi_K = eigve.sel(ky=K[1], method='nearest')`, then `psi_K.sel(band=0)`. `potentials` and `templates` must have the same 2 × 2 shape as `plots`.
</details>

In [ ]:
# Exercise 4.1: your code here
# psi_K = eigve.sel(ky=..., method='nearest')
# psi0, psi1 = psi_K.sel(band=0), psi_K.sel(band=1)
# fig, axs = plot_eigenvector(
#     plots = [[..., ...], [..., ...]],
#     potentials = [[..., ...], [..., ...]],
#     templates = [[..., ...], [..., ...]],
# )
# plt.show()

In [ ]:
# Solution to exercise 4.1
psi_K = eigve.sel(ky=K[1], method='nearest')
psi0, psi1 = psi_K.sel(band=0), psi_K.sel(band=1)

fig, axs = plot_eigenvector(
    plots = [[abs(psi0)**2, abs(psi1)**2],
             [xr.ufuncs.angle(psi0), xr.ufuncs.angle(psi1)]],
    potentials = [[honeycomb, honeycomb], [honeycomb, honeycomb]],
    templates = [['amplitude', 'amplitude'], ['phase', 'phase']],
)
for ax, title in zip(np.ravel(axs), [r'$|\psi_0(K)|^2$', r'$|\psi_1(K)|^2$', r'arg $\psi_0(K)$', r'arg $\psi_1(K)$']):
    ax.set_title(title)
plt.show()
# With delta != 0 the K-point states are sublattice polarized: band 0 sits on the deeper site and
# band 1 on the shallower one. Flipping the sign of delta swaps the two sublattices.

## 5. Cuts through the eigenvectors

As for the potential, a single cartesian axis gives a cut, with the other axis on a slider. The potential given in `potentials` is drawn dashed, against its own y-axis on the right, since both quantities have very different scales. A `cart_axes` matrix mixes maps and cuts in the same figure.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(eigve)**2, abs(eigve)**2]],
    potentials = [[honeycomb, honeycomb]],
    templates = [['amplitude', None]],
    cart_axes = [[[0, 1], [0]]],
)
plt.show()

### ✏️ Exercise 5.1

At the $\Gamma$ point (`ky = 0`), plot **cuts along `x`** of the real part $\mathrm{Re}\,\psi$ of **band 0** and of **band 1**, side by side, with the potential overlaid. Put the `y` slider close to 0, so that the cut passes through both sites.

Which band has the same sign on both sites (a *bonding* state), and which one changes sign between them (*antibonding*)?

<details><summary>Hint</summary>

A single list `cart_axes=[0]` applies to every subplot. Without templates, cuts get simple default line styles.
</details>

In [ ]:
# Exercise 5.1: your code here
# psi_G = eigve.sel(ky=..., method='nearest')
# fig, axs = plot_eigenvector(
#     plots = [[..., ...]],
#     potentials = [[honeycomb, honeycomb]],
#     cart_axes = ...,
# )
# plt.show()

In [ ]:
# Solution to exercise 5.1
psi_G = eigve.sel(ky=0, method='nearest')
fig, axs = plot_eigenvector(
    plots = [[psi_G.real.sel(band=0), psi_G.real.sel(band=1)]],
    potentials = [[honeycomb, honeycomb]],
    cart_axes = [0],
)
axs[0][0].set_title(r'Re $\psi_0(\Gamma)$, cut along x')
axs[0][1].set_title(r'Re $\psi_1(\Gamma)$, cut along x')
plt.show()
# Band 0 keeps the same sign on both sites (bonding), band 1 changes sign between them (antibonding).

## 6. Eigenvalues and eigenvectors together

`dashboard` puts the band structure next to maps of the eigenvectors, all driven by the same sliders. A red dashed line marks the current position along the dimension the bands are plotted against, here `ky`. The `eigveplots` matrix sets the layout of the maps, like `plots` in `plot_eigenvector`. A single `template` styles all the maps, see [section 7](#Styling-the-dashboard).

In [ ]:
from bloch_schrodinger.plotting import dashboard

fig, ax = dashboard(
    eigva, 'ky',
    eigveplots = [[abs(eigve)**2]],
    potential = honeycomb,
    template = 'amplitude',
    titles = [[r'$|\psi|^2$']],
)
plt.show()

### ✏️ Exercise 6.1

Build a dashboard restricted to the **two lowest bands**:

- the band structure against `ky`, with band 0 in blue (`'C0'`) and band 1 in red (`'C3'`),
- two maps: $\mathrm{Re}\,\psi$ and $\mathrm{Im}\,\psi$, both with the `'real'` preset, and titled.

Move the `ky` slider and follow the red marker along the bands.

<details><summary>Hint</summary>

Select `band=[0, 1]` **both** in the eigenvalues and in the eigenvectors. Then the line colors in `linekws` and the `band` slider match.
</details>

In [ ]:
# Exercise 6.1: your code here
# low = eigve.sel(band=[0, 1])
# fig, ax = dashboard(
#     eigva.sel(...), 'ky',
#     eigveplots = [[..., ...]],
#     potential = honeycomb,
#     template = ...,
#     titles = [[..., ...]],
#     linekws = [{...}, {...}],
# )
# plt.show()

In [ ]:
# Solution to exercise 6.1
low = eigve.sel(band=[0, 1])
fig, ax = dashboard(
    eigva.sel(band=[0, 1]), 'ky',
    eigveplots = [[low.real, low.imag]],
    potential = honeycomb,
    template = 'real',
    titles = [[r'Re$(\psi)$', r'Im$(\psi)$']],
    linekws = [{'color': 'C0'}, {'color': 'C3'}],
)
plt.show()

`energy_levels` draws a cut through the potential, and places the profile of each mode along that cut at its energy. The `offset` and `cut rotation` sliders move the cut line in the plane. Passing the tiled potential shows the lattice around the unit cell. The eigenvectors are only defined over a single unit cell, so their profiles stop at its edges.

In [ ]:
from bloch_schrodinger.plotting import energy_levels

fig, ax = energy_levels(eigva, eigve.real, tiled)
plt.show()

### ✏️ Exercise 6.2

Use `energy_levels` to draw the **densities** $|\psi|^2$ of the **two lowest bands** only, on top of the tiled potential. Make the mode profiles twice as tall as the default (`frac` is the fraction of the plot that each profile takes, by default `0.05`).

Rotate the cut with the `cut rotation` slider. What happens to the profiles when the cut no longer passes through the sites?

In [ ]:
# Exercise 6.2: your code here
# fig, ax = energy_levels(eigva.sel(...), ..., tiled, frac=...)
# plt.show()

In [ ]:
# Solution to exercise 6.2
fig, ax = energy_levels(
    eigva.sel(band=[0, 1]),
    abs(eigve.sel(band=[0, 1]))**2,
    tiled,
    frac = 0.1,
)
plt.show()
# When the cut line misses the sites, it only crosses the tails of the modes, so the profiles become small.

## 7. Styling plots with templates

Templates are how the style of `plot_eigenvector` plots is controlled. **A template is a plain dictionary** of instructions for one layer of a subplot. Each subplot of `plot_eigenvector` has up to three layers, each with its own template:

- the **colormesh** of the field given in `plots`, drawn with `pcolormesh`,
- the **contours** of the potential given in `potentials`, drawn with `contour`,
- the **arrows** given in `quivers`, drawn with `quiver`.

Each entry of the `templates` matrix sets the templates of one subplot. It can take the following forms:

| Entry | Colormesh | Contours | Arrows |
|---|---|---|---|
| `None` | matplotlib defaults, no colorbar | matplotlib defaults | matplotlib defaults |
| `'amplitude'` (a preset name) | `cmesh_tmpl('amplitude')` | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `my_dict` | `my_dict` | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `(c,)` | `c`, a name or a dict | `contour_tmpl(ncontours)` | `quiver_tmpl()` |
| `(c, ct)` | `c` | `ct`, a dict | `quiver_tmpl()` |
| `(c, ct, q)` | `c` | `ct` | `q`, a dict |

The functions `cmesh_tmpl`, `contour_tmpl` and `quiver_tmpl` return the default templates, so printing them shows what a template contains:

In [ ]:
from bloch_schrodinger.plotting import cmesh_tmpl, contour_tmpl, quiver_tmpl

print(cmesh_tmpl('amplitude'))
print(contour_tmpl())
print(quiver_tmpl())

### The keys of a template

**Colormesh templates** (the first entry):
- `fkwargs`: keyword arguments passed to `pcolormesh`, e.g. `cmap`, `norm`, `shading` or `vmin`/`vmax`. A `norm` can be given as a function returning a norm, e.g. `lambda: colors.LogNorm()`. Each subplot then builds its own norm, instead of all of them sharing, and rescaling, the same one.
- `autoscale`: if `True`, the color range follows the data each time a slider moves. If `False`, it stays where it started.
- `clim`: a pair `(vmin, vmax)` fixing the color range.
- `colorbar`: without this key, or with `None`, there is no colorbar. Otherwise, a dictionary with the optional keys:
    - `kwargs`, passed to `Figure.colorbar` (e.g. `label`, `format`, `orientation`),
    - `cax`, the position of the colorbar, passed to `append_axes` (default `{'position': 'right', 'size': '5%', 'pad': 0.05}`),
    - `ticks` and `tickslabel`, to set the ticks and their labels by hand.
- `slider_start`: where the sliders start, `'left'` (default), `'mid'` or `'right'`.

**Contour templates** (the second entry): `fkwargs` is passed to `contour`, e.g. `levels`, `colors` or `cmap`, `linewidths` and `linestyles`. The `ncontours` argument of `plot_eigenvector` is a shortcut that sets the number of levels of the default contour template.

**Quiver templates** (the third entry): `fkwargs` is passed to `quiver`, and `density` keeps only one arrow every `density` grid points.

**Line templates**, for cuts (`cart_axes` with a single axis): only `fkwargs`, passed to `plot`, `autoscale` (of the y-axis, default `True`) and `slider_start` are used. The presets are made for maps: their `cmap` and `norm` keywords make `plot` fail, so style cuts with a dictionary or `None`.

Two more pitfalls:
- The contour and quiver entries of a tuple must be dictionaries, not preset names or `None`. To draw the contours with matplotlib's defaults, give `{}`. To hide them, give `None` in `potentials` instead.
- `get_template` returns templates in an older format, which `plot_eigenvector` does not read. Use `cmesh_tmpl` instead.

### The presets

The five presets returned by `cmesh_tmpl` are:
- `'amplitude'`: a sequential colormap on a linear scale, for positive fields such as a density.
- `'amplitude - log'`: the same on a log scale, to see the tails and the nodes of a mode.
- `'real'`: a diverging colormap centered on zero, for signed fields such as the real part of a mode.
- `'real - log'`: the same on a symmetric log scale.
- `'phase'`: a cyclic colormap fixed to $[-\pi, \pi]$, with ticks labelled in units of $\pi$.

Here are four of them on the same eigenvector. The log scale reveals the node of the wavefunction between the two sites, which is invisible on the linear scale. `'real - log'` gets its own example below.

In [ ]:
density = abs(eigve)**2

fig, axs = plot_eigenvector(
    plots = [[density, density], [eigve.real, xr.ufuncs.angle(eigve)]],
    potentials = [[honeycomb, honeycomb], [honeycomb, honeycomb]],
    templates = [['amplitude', 'amplitude - log'], ['real', 'phase']],
)
for ax, title in zip(np.ravel(axs), ["'amplitude'", "'amplitude - log'", "'real'", "'phase'"]):
    ax.set_title(title)
plt.show()

### ✏️ Exercise 7.1 (choosing a preset)

Choose the right preset for each of these three quantities, and plot them in a 1 × 3 figure:

1. the imaginary part $\mathrm{Im}\,\psi$, a signed field,
2. the modulus $|\psi|$ (not squared), on a **log** scale,
3. the phase $\arg\psi$.

Move `ky` to $K$ (`ky ≈ 2.0`) and compare the phase map with the imaginary part.

In [ ]:
# Exercise 7.1: your code here
# fig, axs = plot_eigenvector(
#     plots = [[eigve.imag, abs(eigve), xr.ufuncs.angle(eigve)]],
#     potentials = [[honeycomb, honeycomb, honeycomb]],
#     templates = [[..., ..., ...]],
# )
# plt.show()

In [ ]:
# Solution to exercise 7.1
fig, axs = plot_eigenvector(
    plots = [[eigve.imag, abs(eigve), xr.ufuncs.angle(eigve)]],
    potentials = [[honeycomb, honeycomb, honeycomb]],
    templates = [['real', 'amplitude - log', 'phase']],
)
for ax, title in zip(axs[0], [r'Im$(\psi)$', r'$|\psi|$, log', r'arg$(\psi)$']):
    ax.set_title(title)
plt.show()

### Modifying a preset

Since presets are plain dictionaries, the simplest way to get a custom style is to take a preset and change some of its keys.

On the left, we plot the real part of the states at the $\Gamma$ point with `'real - log'`. By default, this preset has a linear region of only $10^{-12}$ around zero, so its colorbar gets ticks for every decade down to $10^{-12}$. Here we widen the linear region with a new norm, given as a function. The log scale shows the small tails of the wavefunction much more clearly than `'real'` does. Move the `band` slider to see the sign changes of the higher bands.

On the right, we give `'amplitude'` another colormap and a label. We also fix its color range and turn autoscaling off, so that densities can be compared between bands and k-points while moving the sliders.

In [ ]:
from matplotlib import colors

real_log = cmesh_tmpl('real - log')
real_log['fkwargs']['norm'] = lambda: colors.SymLogNorm(linthresh=1e-2)

fixed_density = cmesh_tmpl('amplitude')
fixed_density['fkwargs']['cmap'] = 'magma'
fixed_density['autoscale'] = False
fixed_density['clim'] = (0, 3)
fixed_density['colorbar']['kwargs'] = {'label': r'$|\psi|^2$', 'format': '{x:.1f}'}

psi_gamma = eigve.sel(ky=0, method='nearest') # the states at the Gamma point

fig, axs = plot_eigenvector(
    plots = [[psi_gamma.real, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[real_log, fixed_density]],
)
plt.show()

### ✏️ Exercise 7.2 (modifying a preset)

Start from two presets and change them:

- **`'phase'`**: use the `'hsv'` colormap, and label the colorbar `r'arg$(\psi)$'`.
- **`'amplitude - log'`**: fix the color range to $(10^{-4}, 3)$ and turn autoscaling off.

Plot the phase and the density `density` side by side with these two templates. Move the `band` and `ky` sliders, and check that the log color range stays fixed.

<details><summary>Hint</summary>

`cmesh_tmpl(name)` returns a new dictionary each time, so you can modify it freely. Print it to see which keys to change: the colorbar label is in `tmpl['colorbar']['kwargs']`.
</details>

In [ ]:
# Exercise 7.2: your code here
# my_phase = cmesh_tmpl('phase')
# my_phase['fkwargs'][...] = ...
# my_phase['colorbar']['kwargs'][...] = ...
#
# my_log = cmesh_tmpl('amplitude - log')
# my_log[...] = ...
# my_log[...] = ...
#
# fig, axs = plot_eigenvector(
#     plots = [[xr.ufuncs.angle(eigve), density]],
#     potentials = [[honeycomb, honeycomb]],
#     templates = [[my_phase, my_log]],
# )
# plt.show()

In [ ]:
# Solution to exercise 7.2
my_phase = cmesh_tmpl('phase')
my_phase['fkwargs']['cmap'] = 'hsv'
my_phase['colorbar']['kwargs']['label'] = r'arg$(\psi)$'

my_log = cmesh_tmpl('amplitude - log')
my_log['autoscale'] = False
my_log['clim'] = (1e-4, 3)

fig, axs = plot_eigenvector(
    plots = [[xr.ufuncs.angle(eigve), density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[my_phase, my_log]],
)
plt.show()

### Writing a template from scratch

A template only needs the keys you want to set. Here the field is drawn with a matplotlib colormap and smooth `gouraud` shading. The colorbar is horizontal, below the map, and the sliders start in the middle of their range.

In [ ]:
smooth_real = {
    'fkwargs': {'cmap': 'RdBu_r', 'shading': 'gouraud', 'norm': lambda: colors.CenteredNorm()},
    'autoscale': True,
    'colorbar': {
        'kwargs': {'orientation': 'horizontal', 'label': r'Re$(\psi)$'},
        'cax': {'position': 'bottom', 'size': '5%', 'pad': 0.3},
    },
    'slider_start': 'mid',
}

fig, axs = plot_eigenvector(
    plots = [[eigve.real]],
    potentials = [[honeycomb]],
    templates = [[smooth_real]],
)
plt.show()

### ✏️ Exercise 7.3 (a template from scratch)

Write a colormesh template **from scratch** for the density `density`, with:

- the `'viridis'` colormap and `'gouraud'` shading,
- a fixed color range $(0, 3)$, with no autoscaling,
- a **horizontal colorbar above the map**, with the label $|\psi|^2$ and its ticks on top,
- sliders that start at the **right** end of their range.

<details><summary>Hint</summary>

The colorbar position goes in `'cax'`, e.g. `{'position': 'top', 'size': '5%', 'pad': 0.1}`. The colorbar's own options go in `'kwargs'`: `orientation`, `label` and `ticklocation`.
</details>

In [ ]:
# Exercise 7.3: your code here
# viridis_top = {
#     'fkwargs': {...},
#     'autoscale': ...,
#     'clim': ...,
#     'colorbar': {
#         'kwargs': {...},
#         'cax': {...},
#     },
#     'slider_start': ...,
# }
# fig, axs = plot_eigenvector(plots=[[density]], potentials=[[honeycomb]], templates=[[viridis_top]])
# plt.show()

In [ ]:
# Solution to exercise 7.3
viridis_top = {
    'fkwargs': {'cmap': 'viridis', 'shading': 'gouraud'},
    'autoscale': False,
    'clim': (0, 3),
    'colorbar': {
        'kwargs': {'orientation': 'horizontal', 'label': r'$|\psi|^2$', 'ticklocation': 'top'},
        'cax': {'position': 'top', 'size': '5%', 'pad': 0.1},
    },
    'slider_start': 'right',
}

fig, axs = plot_eigenvector(
    plots = [[density]],
    potentials = [[honeycomb]],
    templates = [[viridis_top]],
)
plt.show()

### Styling the contours and the arrows

The second entry of a tuple styles the contours of the potential. `contour_tmpl` accepts either a number of levels or the levels themselves. A dictionary gives full control, here with contours colored by a colormap. The arrows are styled the same way through the third entry, as done for the force field in section 1.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[density, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[
        ('amplitude', contour_tmpl(np.linspace(-15, 30, 10))), # 10 levels at fixed energies
        ('amplitude', {'fkwargs': {'levels': 8, 'cmap': 'inferno', 'linewidths': 1}}),
    ]],
)
plt.show()

### ✏️ Exercise 7.4 (contours and arrows)

Overlay the dipole force on the density of the eigenstates, inside the **single unit cell**:

1. Compute `Fx_cell` and `Fy_cell` from `honeycomb.V`, exactly as `Fx`, `Fy` were computed from `tiled.V` in section 1. The arrows must live on the same grid as the map they are drawn on.
2. Plot `density` with the `'amplitude'` preset, **6 solid black** contour lines of width 1, and the force as **orange** arrows (`'C1'`), one every 6 grid points.

<details><summary>Hint</summary>

`A_inv` from section 1 still holds. The tuple is `('amplitude', contour_dict, quiver_dict)`, and contour styles go in `fkwargs`: `levels`, `colors`, `linewidths`, `linestyles`.
</details>

In [ ]:
# Exercise 7.4: your code here
# dV1_cell = honeycomb.V.differentiate(...)
# dV2_cell = ...
# Fx_cell = -(A_inv[0, 0] * dV1_cell + A_inv[0, 1] * dV2_cell)
# Fy_cell = ...
#
# black_contours = {'fkwargs': {...}}
# orange_arrows = {'fkwargs': {...}, 'density': ...}
#
# fig, axs = plot_eigenvector(
#     plots = [[density]],
#     potentials = [[honeycomb]],
#     templates = [[('amplitude', black_contours, orange_arrows)]],
#     quivers = [[(Fx_cell, Fy_cell)]],
# )
# plt.show()

In [ ]:
# Solution to exercise 7.4
dV1_cell = honeycomb.V.differentiate('a1')
dV2_cell = honeycomb.V.differentiate('a2')
Fx_cell = -(A_inv[0, 0] * dV1_cell + A_inv[0, 1] * dV2_cell)
Fy_cell = -(A_inv[1, 0] * dV1_cell + A_inv[1, 1] * dV2_cell)

black_contours = {'fkwargs': {'levels': 6, 'colors': 'k', 'linewidths': 1, 'linestyles': 'solid'}}
orange_arrows = {
    'fkwargs': {'color': 'C1', 'pivot': 'mid', 'angles': 'xy', 'scale_units': 'xy', 'scale': 200, 'width': 0.008},
    'density': 6,
}

fig, axs = plot_eigenvector(
    plots = [[density]],
    potentials = [[honeycomb]],
    templates = [[('amplitude', black_contours, orange_arrows)]],
    quivers = [[(Fx_cell, Fy_cell)]],
)
plt.show()

### Styling cuts

Cuts are styled with line templates. A matrix of templates and a matrix of `cart_axes` give each subplot its own style. Here a map with a preset is next to a cut, drawn as a thick red line, whose y-axis stays fixed while the sliders move.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[density, density]],
    potentials = [[honeycomb, honeycomb]],
    templates = [['amplitude', {'fkwargs': {'color': 'C3', 'linewidth': 2}, 'autoscale': False}]],
    cart_axes = [[[0, 1], [0]]],
)
plt.show()

### ✏️ Exercise 7.5 (styling cuts)

Make a 1 × 2 figure of cuts:

- left: a cut of `density` along `x`, drawn as a **dashed green** line (`'C2'`), with a **fixed** y-axis,
- right: a cut of `eigve.real` along `y`, drawn as a **thick orange** line (`'C1'`, width 2), whose y-axis follows the sliders.

Remember that the presets are made for maps: style cuts with a plain dictionary.

In [ ]:
# Exercise 7.5: your code here
# fig, axs = plot_eigenvector(
#     plots = [[density, eigve.real]],
#     potentials = [[honeycomb, honeycomb]],
#     templates = [[{...}, {...}]],
#     cart_axes = [[..., ...]],
# )
# plt.show()

In [ ]:
# Solution to exercise 7.5
fig, axs = plot_eigenvector(
    plots = [[density, eigve.real]],
    potentials = [[honeycomb, honeycomb]],
    templates = [[
        {'fkwargs': {'color': 'C2', 'linestyle': '--'}, 'autoscale': False},
        {'fkwargs': {'color': 'C1', 'linewidth': 2}, 'autoscale': True},
    ]],
    cart_axes = [[[0], [1]]],
)
axs[0][0].set_title(r'$|\psi|^2$ along x')
axs[0][1].set_title(r'Re$(\psi)$ along y')
plt.show()

### Styling the dashboard

`dashboard` takes one template, shared by all its maps. It accepts a preset name, a colormesh template dict, or a tuple `(colormesh, contour)` to style the potential contours as well. There is no quiver entry, since the dashboard draws no arrows. The maps never get a colorbar, since they share the sliders but not their color scales.

In [ ]:
fig, ax = dashboard(
    eigva, 'ky',
    eigveplots = [[density, abs(psi_gamma)**2]], # the second map does not follow the ky slider
    potential = honeycomb,
    template = (fixed_density, contour_tmpl(8)), # the fixed color range makes both maps comparable
    titles = [[r'$|\psi|^2$', r'$|\psi|^2$ at $\Gamma$']],
)
plt.show()

### ✏️ Exercise 7.6 (styling the dashboard)

Make a dashboard of the full band structure against `ky`, with a single map of `density` on a **log scale**. The contours of the potential should be **5 solid black** lines.

In [ ]:
# Exercise 7.6: your code here
# fig, ax = dashboard(
#     eigva, 'ky',
#     eigveplots = [[density]],
#     potential = honeycomb,
#     template = (..., {...}),
#     titles = [[...]],
# )
# plt.show()

In [ ]:
# Solution to exercise 7.6
fig, ax = dashboard(
    eigva, 'ky',
    eigveplots = [[density]],
    potential = honeycomb,
    template = ('amplitude - log', {'fkwargs': {'levels': 5, 'colors': 'k', 'linewidths': 0.8, 'linestyles': 'solid'}}),
    titles = [[r'$|\psi|^2$, log scale']],
)
plt.show()

## 8. Wannier functions

Wannier functions are plotted with the same functions as eigenvectors: they are fields over the lattice, with the band `n` and the lattice parameters as extra dimensions. How they are computed is the subject of the [Wannier](Wannier.ipynb) tutorial; here we compute the two lowest-band Wannier functions of our honeycomb and look at them.

One detail matters for the plots. Which site is deeper depends on the sign of `delta`, so the **centres** passed to `Wannier.solve` are chosen per parameter point: a DataArray with dims `('n', 'coord')` plus any of the potential's parameters. Here band `n = 0` is always seeded on the deeper site, so `n = 0` is the more strongly bound function at every point. `compute_wannier` returns the Wannier functions over several unit cells, together with the potential tiled over the same cells, which is what we draw them on.

In [ ]:
from bloch_schrodinger.wannier import Wannier

# Reciprocal vectors of this lattice: the Brillouin zone the k-mesh covers
b1 = np.array([-1, 3**0.5]) * 2*np.pi/3/a
b2 = np.array([1, 3**0.5]) * 2*np.pi/3/a

# The two sites sit at (-a/2, 0) and (a/2, 0), i.e. at lattice coordinates (-1/6, -1/6) and
# (1/6, 1/6). Seed n = 0 on whichever of them is deeper at each (s, delta).
left_deeper = honeycomb.V.interp(a1=-1/6, a2=-1/6) < honeycomb.V.interp(a1=1/6, a2=1/6)
pair = xr.DataArray([[-a/2, 0], [a/2, 0]], dims=('n', 'coord'))
centers = xr.where(left_deeper, pair, pair[::-1].values)   # dims (s, delta, n, coord)

wan = Wannier(honeycomb, hbar**2/2/m, rec_vecs=[b1, b2], resolution=(12, 12))
U_mnk = wan.solve(n_wannier=2, centers=centers, parallel=True, blockwargs={'E_lim': 300}, seed=0)
wpot, wannier = wan.compute_wannier(U_mnk, bounds=[[-2, 3], [-2, 3]])

print(wannier.dims)

### Maps, with the sites marked

`plot_eigenvector` draws the densities $|w_n|^2$ like any eigenvector. `show_minima=True` marks the local minima of each subplot's potential, i.e. the lattice sites, and moves them with the sliders, as `Potential.plot` does. It reads only the sliders of the potential's own parameters, so `s` and `delta` move the sites while the band index of the plotted field does not concern them. Move `delta` through zero: the deeper site, and with it $w_0$, jumps from one sublattice to the other.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(wannier.sel(n=0))**2, abs(wannier.sel(n=1))**2]],
    potentials = [[wpot, wpot]],
    templates = [['amplitude', 'amplitude']],
    show_minima = True,
)
axs[0][0].set_title('$|w_0|^2$ (deeper site)')
axs[0][1].set_title('$|w_1|^2$')
for ax in axs[0]:
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
plt.show()

### ✏️ Exercise 8.1

Plot the **real parts** of $w_0$ and $w_1$ side by side, with the `'real'` preset and the lattice sites marked as **white crosses**. Zoom in on $x, y \in [-3, 3]$.

Check two things with the sliders:

- the two functions sit on neighbouring sites of different sublattices,
- the largest lobe of each function is positive or negative. (Does the sign carry any physical meaning?)

<details><summary>Hint</summary>

`minima_kwargs` is passed to `scatter`, so `{'color': 'white', 'marker': 'x', 's': 40}` gives white crosses. (Use `color`, not `c`: the default style already sets `color`.)
</details>

In [ ]:
# Exercise 8.1: your code here
# fig, axs = plot_eigenvector(
#     plots = [[wannier.real.sel(n=0), ...]],
#     potentials = [[wpot, wpot]],
#     templates = [[..., ...]],
#     show_minima = ...,
#     minima_kwargs = {...},
# )
# for ax in axs[0]:
#     ax.set_xlim(...)
#     ax.set_ylim(...)
# plt.show()

In [ ]:
# Solution to exercise 8.1
fig, axs = plot_eigenvector(
    plots = [[wannier.real.sel(n=0), wannier.real.sel(n=1)]],
    potentials = [[wpot, wpot]],
    templates = [['real', 'real']],
    show_minima = True,
    minima_kwargs = {'color': 'white', 'marker': 'x', 's': 40},
)
axs[0][0].set_title(r'Re$(w_0)$')
axs[0][1].set_title(r'Re$(w_1)$')
for ax in axs[0]:
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
plt.show()
# The global sign (more generally, the global phase) of a Wannier function is a gauge choice: it has no
# physical meaning. Only the relative signs of its lobes on the different sites do.

### Tails and signs

On a linear scale a Wannier function looks like a single blob. Its tails, the small amplitude it has on the neighbouring sites, are what tunnelling and nearest-neighbour interactions are made of, and they only show on a log scale. Fixing the colour range (`autoscale = False`, `clim`) keeps the scale put while the sliders move, so the tails can be compared across `s`: they shrink fast as the lattice deepens.

`show_minima` also takes a matrix shaped like `plots`, to mark the sites on some subplots only. Here the sign structure of $\mathrm{Re}(w_0)$ on the left is left unmarked.

In [ ]:
log_density = cmesh_tmpl('amplitude - log')
log_density['autoscale'] = False
log_density['clim'] = (1e-6, 1)

fig, axs = plot_eigenvector(
    plots = [[wannier.real.sel(n=0), abs(wannier.sel(n=0))**2]],
    potentials = [[wpot, wpot]],
    templates = [['real', log_density]],
    show_minima = [[False, True]],
    minima_kwargs = {'s': 12},
)
axs[0][0].set_title(r'Re$(w_0)$')
axs[0][1].set_title(r'$|w_0|^2$, log scale')
for ax in axs[0]:
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
plt.show()

### ✏️ Exercise 8.2

Compare the **tails** of the two Wannier functions. Plot $|w_0|^2$ and $|w_1|^2$ side by side, on a log scale with the **same fixed** color range $(10^{-8}, 1)$, and mark the sites on both panels. Zoom in on $x, y \in [-5, 5]$.

At `delta = ±0.2`, which function has the longer tails? Does this fit with which site each one sits on?

In [ ]:
# Exercise 8.2: your code here
# tails = cmesh_tmpl(...)
# tails['autoscale'] = ...
# tails['clim'] = ...
#
# fig, axs = plot_eigenvector(
#     plots = [[..., ...]],
#     potentials = [[wpot, wpot]],
#     templates = [[tails, tails]],
#     show_minima = ...,
# )
# ...
# plt.show()

In [ ]:
# Solution to exercise 8.2
tails = cmesh_tmpl('amplitude - log')
tails['autoscale'] = False
tails['clim'] = (1e-8, 1)

fig, axs = plot_eigenvector(
    plots = [[abs(wannier.sel(n=0))**2, abs(wannier.sel(n=1))**2]],
    potentials = [[wpot, wpot]],
    templates = [[tails, tails]],
    show_minima = True,
)
axs[0][0].set_title(r'$|w_0|^2$ (deeper site)')
axs[0][1].set_title(r'$|w_1|^2$ (shallower site)')
for ax in axs[0]:
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
plt.show()
# w_1 sits on the shallower site: it is less tightly bound, so its tails reach further.

### Maps and cuts together

A `cart_axes` matrix mixes maps and cuts, as for the eigenvectors. With `show_minima=True` the sites are marked where they can be: on the map, which shows the whole plane. A cut through a 2D potential has no reason to pass through its minima, so it is left unmarked rather than refused. The `n` slider switches between the two functions in both panels at once.

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(wannier)**2, abs(wannier)**2]],
    potentials = [[wpot, wpot]],
    templates = [['amplitude', None]],
    cart_axes = [[[0, 1], [0]]],
    show_minima = True,
)
plt.show()

### ✏️ Exercise 8.3

Make a **1 × 3** figure of $|w|^2$: the map with the sites marked, a cut along `x`, and a cut along `y`. Keep the `n` slider, so that it switches between $w_0$ and $w_1$ in all three panels at once. Style the two cuts as red lines.

In [ ]:
# Exercise 8.3: your code here
# red = {'fkwargs': {...}}
# fig, axs = plot_eigenvector(
#     plots = [[abs(wannier)**2] * 3],
#     potentials = [[wpot] * 3],
#     templates = [[..., red, red]],
#     cart_axes = [[..., ..., ...]],
#     show_minima = ...,
# )
# plt.show()

In [ ]:
# Solution to exercise 8.3
red = {'fkwargs': {'color': 'C3'}}
fig, axs = plot_eigenvector(
    plots = [[abs(wannier)**2] * 3],
    potentials = [[wpot] * 3],
    templates = [['amplitude', red, red]],
    cart_axes = [[[0, 1], [0], [1]]],
    show_minima = True,   # only the map is marked; the cuts are skipped
)
for ax, title in zip(axs[0], ['map', 'cut along x', 'cut along y']):
    ax.set_title(title)
plt.show()

### Quantities derived from the Wannier functions

Like the eigenvalues, anything computed from the Wannier functions with ordinary xarray operations is plotted with `plot_cuts`. Here the spread $\langle r^2\rangle - \langle \mathbf r\rangle^2$ of each function, against `delta`, one line per band with `groupby=['n']`. At `delta = 0` the two sites are equivalent and so are the two functions. Any imbalance makes $w_0$, on the deeper site, tighter, and $w_1$ broader.

In [ ]:
w_density = abs(wannier)**2 / (abs(wannier)**2).sum(['a1', 'a2'])
mean_x = (w_density * wannier.x).sum(['a1', 'a2'])
mean_y = (w_density * wannier.y).sum(['a1', 'a2'])
spread = ((w_density * (wannier.x**2 + wannier.y**2)).sum(['a1', 'a2'])
          - mean_x**2 - mean_y**2).rename('spread')

fig, ax = plot_cuts(
    spread, 'delta', groupby=['n'],
    linekws = [{'color': 'C0', 'label': '$w_0$ (deeper site)'}, {'color': 'C1', 'label': '$w_1$'}],
)
ax.legend()
plt.show()

### ✏️ Exercise 8.4

The overlap of the densities of two neighbouring Wannier functions,

$$O = \int |w_0(\mathbf r)|^2\, |w_1(\mathbf r)|^2 \, d^2 r ,$$

sets the scale of the **nearest-neighbour density–density interaction** for contact interactions. Compute it with `w_density` from the previous cell (sum over `a1` and `a2`, arbitrary units). Plot it against `delta`, with **one line per `s`** and a **log** y-axis.

- By what factor does the overlap drop from `s = 3` to `s = 9`?
- Is the overlap largest or smallest at `delta = 0`? Why? (Think back to the spreads plotted above.)

<details><summary>Hint</summary>

`plot_cuts` rescales the y-axis linearly unless you give both `ymin` and `ymax`. For a log axis, pass them from the data, e.g. `ymin=float(overlap.min()) / 2`, and call `ax.set_yscale('log')` after plotting.
</details>

In [ ]:
# Exercise 8.4: your code here
# overlap = (w_density.sel(n=0) * w_density.sel(n=1)).sum([...]).rename('density overlap')
# fig, ax = plot_cuts(
#     overlap, 'delta', groupby=[...],
#     linekws = [...],
#     ymin = ..., ymax = ...,
# )
# ax.set_yscale('log')
# ax.legend()
# plt.show()

In [ ]:
# Solution to exercise 8.4
overlap = (w_density.sel(n=0) * w_density.sel(n=1)).sum(['a1', 'a2']).rename('density overlap')
fig, ax = plot_cuts(
    overlap, 'delta', groupby=['s'],
    linekws = [{'label': f's = {v:g}', 'marker': 'o'} for v in overlap.s.values],
    ymin = float(overlap.min()) / 2, ymax = float(overlap.max()) * 2,
)
ax.set_yscale('log')
ax.legend()
plt.show()
print(overlap.to_pandas())
# The overlap drops by a factor of about 3 from s = 3 to s = 9, as the functions get tighter.
# It is smallest at delta = 0: any imbalance broadens w_1 (on the shallower site), and its tail
# reaches further onto the neighbouring site of w_0.

## 9. The lattice geometry: snapped sites and a schematic overlay

`find_minima` finds the sites of the lattice numerically, so their positions are only as precise as the grid: on our 48 × 48 cell the minima come out a few hundredths off. When the **geometry** of the lattice is known, the sites sit at fixed points, whatever the depths. In a honeycomb they sit on the C3-symmetric points of its triangular Bravais lattice. For figures we want those exact points, and we want the lattice drawn as a schematic: labelled circles at the sites, the bonds between them as dashed lines.

The module `bloch_schrodinger.lattice` does this in three steps:

1. A `LatticeGeometry` holds the lattice vectors and the **named sites**, at exact positions.
2. `Potential.find_sites` snaps the minima the potential actually has onto those sites and labels them.
3. The `lattice` option of `Potential.plot`, `plot_eigenvector` and `dashboard`, or `draw_lattice` on an axes of your own, draws the lattice over the map.

### The geometry

A `LatticeGeometry` is the pair of lattice vectors and a set of named sites, given in fractional coordinates of the vectors, from an `origin`. Five presets place the sites from the vectors you give, in whatever orientation they have:

| Preset | Lattice vectors | Sites |
|---|---|---|
| `hexagonal` | equal, at 60° or 120° | A at the origin, B and C on the two other C3-symmetric points: a honeycomb occupies two of them, a triangular lattice one, the third is the hexagon centre |
| `square` | equal, perpendicular | A |
| `checkerboard` | the two diagonals of the squares | A, and B at $(a_1 + a_2)/2$ |
| `kagome` | equal, at 60° or 120° | A, B, C on the bond midpoints of a triangular lattice |
| `lieb` | equal, perpendicular | A at the corner, B and C on the edges |

**Sites are labelled by position, not by depth.** A, B and C are fixed points of the geometry. Which of them are wells is the physics: here the depths, in a polarization-engineered lattice the polarization. A well found on the C point is called C.

Our honeycomb's wells sit at $(-a/2, 0)$ and $(a/2, 0)$ (section 8). We put A on the first one, which makes B the second and C the hexagon centre at $(3a/2, 0)$. Setting the geometry as the potential's `geometry` attribute lets the plotting functions find it with `lattice=True`. `tile` (and with it `compute_wannier`) passes it on to the potentials it makes. `tiled` and `wpot` were made before, so they get it by hand. The constructors `optical_honeycomb` and `honeycomb` set the geometry themselves.

In [ ]:
from bloch_schrodinger.lattice import LatticeGeometry, draw_lattice

geom = LatticeGeometry.hexagonal(a1, a2, origin=(-a/2, 0))
honeycomb.geometry = geom
tiled.geometry = geom
wpot.geometry = geom

print(geom)
print('nearest-neighbour distance:', geom.nn_distance(), '= a =', a)

`draw_lattice` draws a geometry on any axes. Here are the five presets on empty axes, with the bonds and the unit cell (the `bonds` and `unit_cell` options, see *Styling* below). By default every site of a geometry is drawn as occupied.

In [ ]:
s3 = 3**0.5
presets = [
    LatticeGeometry.hexagonal([1, 0], [0.5, s3/2]),
    LatticeGeometry.square([1, 0], [0, 1]),
    LatticeGeometry.checkerboard([1, 1], [1, -1]),
    LatticeGeometry.kagome([1, 0], [0.5, s3/2]),
    LatticeGeometry.lieb([1, 0], [0, 1]),
]
fig, axs = plt.subplots(1, 5, figsize=(15, 3.4), layout='constrained')
for ax, g in zip(axs, presets):
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.set_title(g.name)
    draw_lattice(ax, g, bonds=True, unit_cell=True)
plt.show()

### Snapping the minima onto the sites

`find_sites` runs `find_minima` and moves every minimum that lies within `tol` times the nearest-neighbour distance of a site (default 10 %) exactly onto it, with that site's label. A minimum that is near no site is a real finding, not noise. It is **kept where it was found**, labelled `""`, and listed in a warning.

The result is a Dataset along `minima`:
- `x`, `y`: the snapped positions;
- `x_found`, `y_found`: the positions on the grid;
- `distance`: how far apart the two are;
- `label`, `on_site`, and `v`, the potential at the minimum.

In [ ]:
sites = honeycomb.find_sites(selection={'s': 6, 'delta': 0.1})
print(sites[['label', 'x', 'y', 'x_found', 'y_found', 'distance', 'v']].to_dataframe())

The grid put both minima 0.02 off the bond axis; the snapped positions are exact. Without a selection, `find_sites` covers every `(s, delta)` at once, like `find_minima`:

In [ ]:
all_sites = honeycomb.find_sites()
print(all_sites.label.dims, '-> labels found:', sorted(set(all_sites.label.values.ravel()) - {''}))
print('largest correction:', float(all_sites.distance.max()))

### ✏️ Exercise 9.1 (snapping)

At `s = 3`, find the sites of `honeycomb` for `delta = -0.2` and for `delta = +0.2`. For each, print the label of the **deeper** site (the one with the lowest `v`) and the largest snapping correction `distance`. Does the deeper site change sublattice?

*Hint:* `sites.label.values[np.argmin(sites.v.values)]`.

In [ ]:
# Exercise 9.1: your code here
# for d in (-0.2, 0.2):
#     sites = honeycomb.find_sites(selection={'s': ..., 'delta': ...})
#     deeper = ...
#     print(...)

In [ ]:
# Solution to exercise 9.1
for d in (-0.2, 0.2):
    sites = honeycomb.find_sites(selection={'s': 3, 'delta': d})
    deeper = sites.label.values[np.argmin(sites.v.values)]
    print(f'delta = {d:+.1f}: deeper site {deeper}, largest correction {float(sites.distance.max()):.3f}')
# V1 = -(s - delta) E_r is the depth of A, V2 = -(s + delta) E_r that of B:
# the deeper site is A for delta < 0 and B for delta > 0

### The overlay on the plots

`lattice=True` draws the lattice over a map. It works in `Potential.plot`, in `plot_eigenvector` (one value, or a matrix like `show_minima`) and in `dashboard`. The circles are the sites that the potential's minima snap to at the current slider values, so they **follow the sliders**. The overlay fills whatever area the axes show, so it extends past the cell the potential covers, and it is redrawn when you zoom or set new limits. Instead of `True`, a `LatticeGeometry` can be passed directly.

By default only the labelled circles are drawn. Everything else is an option in `lattice_kwargs` (see *Styling* below). A contour template of `None`, as in `('amplitude', None)`, leaves out the potential's contours, so the schematic replaces them. The potential is still used to find the sites.

In [ ]:
fig, ax = tiled.plot(lattice=True)
ax.set_xlim(-3, 3)          # inside the tiled cells: the overlay fills whatever the axes show
ax.set_ylim(-1.5, 1.5)
plt.show()

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[abs(wannier.sel(n=0))**2, abs(wannier.sel(n=1))**2]],
    potentials = [[wpot, wpot]],
    templates = [[('amplitude', None), ('amplitude', None)]],   # no contours: the schematic instead
    lattice = True,
    lattice_kwargs = {'bonds': True},
)
axs[0][0].set_title('$|w_0|^2$ (deeper site)')
axs[0][1].set_title('$|w_1|^2$')
for ax in axs[0]:
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
plt.show()

Which sites are drawn is decided by the potential, at each slider position. `optical_honeycomb` builds the same three-beam lattice from the depths `s1` and `s2` of its two triangular sublattices, and sets its geometry itself: A is the `s2` sublattice, B the `s1` one. With `s2 = 4` and `s1` on a slider:
- at `s1 = 0` the lattice is triangular: only A are wells, and the bonds join A to A;
- once `s1` exceeds about `s2 / 2`, B becomes a well too, and the bonds become the honeycomb.

`empty_sites` shows the other points of the geometry as faint open circles.

In [ ]:
from bloch_schrodinger.potential import optical_honeycomb

s1 = create_parameter('s1', [0, 1, 2.5, 4])
constants, optical = optical_honeycomb(wavelength=2*np.pi/kl, s1=s1, s2=4, resolution=(48, 48))
print(optical.geometry)

optical_tiled = optical.tile([[-1, 2], [-1, 2]])
fig, axs = plot_eigenvector(
    plots = [[optical_tiled.V]],
    potentials = [[optical_tiled]],
    templates = [[('real', None)]],
    lattice = True,
    lattice_kwargs = {'bonds': True, 'empty_sites': True},
)
axs[0][0].set_xlim(-2, 3)
axs[0][0].set_ylim(-2, 2)
plt.show()

### Styling

`lattice_kwargs` takes these keys:

**What is drawn**
- `occupied`: `'minima'` (the default), `'all'`, or a list of labels. With `'minima'`, the occupied sites are those the minima snap to.
- `tol`: the snapping tolerance. Defaults to 0.1.
- `sites` and `labels`: the circles and the letters in them. Both default to True.
- `bonds`: dashed lines between nearest-neighbour occupied sites.
- `empty_sites`: the unoccupied sites, as faint open circles.
- `unit_cell`: the cell at the origin, with $a_1$ and $a_2$ as arrows.
- `offsite`: marks minima that are near no site with a grey ×. Defaults to True.

**How it looks**
- `colors`: per label, a fill colour or a `(fill, edge)` pair.
- `size`: by default the circles scale with the zoom; a number fixes their area, in points², as matplotlib's `scatter` does.
- `fontsize`: the size of the labels.
- `site_kwargs`, `text_kwargs`, `bond_kwargs`, `empty_kwargs`, `offsite_kwargs` and `cell_kwargs`: passed on to matplotlib.

A signed field drawn with a diverging colour map and no contours, as in the polarization maps of a polarization-engineered lattice:

In [ ]:
fig, axs = plot_eigenvector(
    plots = [[eigve.real]],
    potentials = [[honeycomb]],
    templates = [[('real', None)]],
    lattice = True,
    lattice_kwargs = {
        'bonds': True,
        'unit_cell': True,
        'empty_sites': True,
        'colors': {'A': '#c2185b', 'B': ('#9ccc65', '#33691e')},
        'bond_kwargs': {'colors': 'black', 'linewidths': 1.2},
    },
)
axs[0][0].set_xlim(-2, 2.5)
axs[0][0].set_ylim(-1.8, 1.8)
plt.show()

### ✏️ Exercise 9.2 (the overlay)

Plot the **real parts** of $w_0$ and $w_1$ side by side with the `'real'` preset and **no contours**, and draw the lattice over them:
- with the bonds and the empty site C;
- with A in red (`'#d62728'`) and B in light grey (`'#dddddd'`);
- with axes from $-4$ to $4$.

Move `delta` through zero and watch the sign structure follow the deeper site.

In [ ]:
# Exercise 9.2: your code here
# fig, axs = plot_eigenvector(
#     plots = [[wannier.real.sel(n=0), wannier.real.sel(n=1)]],
#     potentials = [[wpot, wpot]],
#     templates = [[('real', ...), ('real', ...)]],
#     lattice = ...,
#     lattice_kwargs = {...},
# )

In [ ]:
# Solution to exercise 9.2
fig, axs = plot_eigenvector(
    plots = [[wannier.real.sel(n=0), wannier.real.sel(n=1)]],
    potentials = [[wpot, wpot]],
    templates = [[('real', None), ('real', None)]],
    lattice = True,
    lattice_kwargs = {'bonds': True, 'empty_sites': True,
                      'colors': {'A': '#d62728', 'B': '#dddddd'}},
)
axs[0][0].set_title('Re $w_0$')
axs[0][1].set_title('Re $w_1$')
for ax in axs[0]:
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
plt.show()

### Your own figure: `draw_lattice`

`draw_lattice(ax, geometry, occupied)` puts the same overlay on any axes, so figures that the plotting functions do not cover can have it too. Set the limits first: the overlay fills them. It returns the overlay object, whose `draw(occupied)` redraws it.

Here both Wannier functions are drawn in one panel, as filled contours in two colours, with the sites their potential has at this point:

In [ ]:
point = {'s': 6, 'delta': 0.1}
occupied = set(wpot.find_sites(selection=point).label.values) - {''}

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for n, cmap in ((0, 'Blues'), (1, 'Greens')):
    w = abs(wannier.sel(n=n).sel(point, method='nearest'))**2
    w = (w / w.max()).transpose('a1', 'a2')
    ax.contourf(w.x, w.y, w, levels=[0.02, 0.1, 0.3, 0.6, 1.01], cmap=cmap, alpha=0.55)
    ax.contour(w.x, w.y, w, levels=[0.02, 0.1, 0.3, 0.6], colors='0.4', linewidths=0.5)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
draw_lattice(ax, geom, occupied=occupied, bonds=True)
ax.set_title('$|w_0|^2$ (blue) and $|w_1|^2$ (green)')
plt.show()

### ✏️ Exercise 9.3 (your own figure)

Remake the two-Wannier-function panel above for `delta = -0.2` (still `s = 6`). Draw the **more strongly bound** function in blue and the other in green, so that the colours always match the depth. Take the occupied sites from `wpot.find_sites`, and add the unit cell.

*Hint:* section 8 seeded `n = 0` on the deeper site at every point.

In [ ]:
# Exercise 9.3: your code here
# point = {'s': 6, 'delta': -0.2}
# occupied = ...
# fig, ax = plt.subplots(figsize=(5.5, 4.5))
# for n, cmap in ((..., 'Blues'), (..., 'Greens')):
#     ...
# draw_lattice(ax, geom, occupied=occupied, bonds=True, unit_cell=...)
# plt.show()

In [ ]:
# Solution to exercise 9.3
point = {'s': 6, 'delta': -0.2}
occupied = set(wpot.find_sites(selection=point).label.values) - {''}

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for n, cmap in ((0, 'Blues'), (1, 'Greens')):        # n = 0 is the more strongly bound one
    w = abs(wannier.sel(n=n).sel(point, method='nearest'))**2
    w = (w / w.max()).transpose('a1', 'a2')
    ax.contourf(w.x, w.y, w, levels=[0.02, 0.1, 0.3, 0.6, 1.01], cmap=cmap, alpha=0.55)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-2.5, 2.5)
ax.set_aspect('equal')
draw_lattice(ax, geom, occupied=occupied, bonds=True, unit_cell=True)
ax.set_title('delta = -0.2: $w_0$ (blue) now sits on A')
plt.show()

### A geometry of your own

Any lattice can be described by its vectors and its named sites in fractional coordinates. Snapping, `find_sites` and the overlay then work as for the presets. For instance, a rectangular lattice with a two-site basis:

In [ ]:
dimer = LatticeGeometry([2, 0], [0, 1], sites={'A': (0, 0), 'B': (0.3, 0.5)}, name='dimer')
print(dimer)
print(dimer.snap([[0.62, 0.47], [1.0, 0.0]], warn=False))

fig, ax = plt.subplots(figsize=(5, 3))
ax.set_xlim(-3, 3)
ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
draw_lattice(ax, dimer, bonds=True, unit_cell=True)
plt.show()

## Summary

| What you want | Call |
|---|---|
| Potential landscape | `pot.plot()`, or `plot_eigenvector([[pot.V]], [[None]], [['real']])` |
| Cut through the potential | `pot.plot(cart_axes=[0])`, or `plot_eigenvector(..., cart_axes=[0])` |
| Band structure, eigenvalues vs. a parameter | `plot_cuts(eigva, 'ky')` |
| Any derived quantity vs. a parameter | `plot_cuts(quantity, 'delta', groupby=[])` |
| Eigenvector maps | `plot_eigenvector([[abs(eigve)**2]], [[pot]], [['amplitude']])` |
| Cuts through eigenvectors | `plot_eigenvector(..., cart_axes=[0])` |
| Bands and eigenvectors side by side | `dashboard(eigva, 'ky', [[abs(eigve)**2]], pot, 'amplitude')` |
| Mode profiles at their energies | `energy_levels(eigva, eigve, pot)` |
| Wannier functions, sites marked | `plot_eigenvector([[abs(wannier)**2]], [[wpot]], [['amplitude']], show_minima=True)` |
| Sites marked on some subplots only | `show_minima=[[True, False]]` |
| Style a subplot | `templates=[['amplitude']]`, `[[(cmesh_tmpl('real'), contour_tmpl(6), quiver_tmpl())]]`, or dictionaries |
| Style a dashboard | `template='amplitude'`, a dict, or `(colormesh, contour)` |
| The lattice geometry, sites at exact positions | `geom = LatticeGeometry.hexagonal(a1, a2, origin=...)`; `pot.geometry = geom` |
| Minima snapped onto the sites, labelled | `pot.find_sites(geom)` |
| The lattice drawn over a map | `plot_eigenvector(..., lattice=True, lattice_kwargs={'bonds': True})`, also in `pot.plot` and `dashboard` |
| A map without the potential's contours | `templates=[[('amplitude', None)]]` |
| The lattice on a figure of your own | `draw_lattice(ax, geom, occupied=['A', 'B'], bonds=True)` |

Every function returns the matplotlib figure and axes, so titles, limits and labels can be adjusted afterwards. For plots that none of these functions cover, the low-level `create_map`, `create_line` and `create_quiver` functions handle the interactivity of a single panel of your own figure. See the [Wannier](Wannier.ipynb) and [PolaritonDispersion](PolaritonDispersion.ipynb) tutorials.